# 10.4 Web Deployment (ONNX.js) — Apply

## Objective

Simulate browser-based ONNX inference in Python. We build the preprocessing,
postprocessing, caching, and configuration logic that would run in a web application
using ONNX Runtime Web (formerly ONNX.js).

**Prerequisites:** `pip install onnx onnxruntime numpy`

## Table of Contents
1. [Setup](#setup)
2. [Exercise 1 — Browser-Style Preprocessing](#ex1)
3. [Exercise 2 — Softmax & Top-K Classification](#ex2)
4. [Exercise 3 — Backend Selection Logic](#ex3)
5. [Exercise 4 — Model Caching Strategy](#ex4)
6. [Exercise 5 — Model Size Optimization for Web](#ex5)
7. [Exercise 6 — JSON-Serializable Inference API](#ex6)
8. [Exercise 7 — Float16 & Quantized Models for Web](#ex7)
9. [Challenge — Web Deployment Manifest Generator](#challenge)
10. [Summary](#summary)

In [ ]:
!pip install onnx onnxruntime numpy -q

<a id='setup'></a>
## Setup

We build a small image classification model using `onnx.helper` to simulate
what would be loaded by ONNX Runtime Web in the browser.

In [ ]:
import os
import time
import json
import hashlib
import numpy as np
from typing import Any, Dict, List, Optional, Tuple

import onnx
from onnx import helper, TensorProto, numpy_helper
import onnxruntime as ort

np.random.seed(42)

def build_web_classifier(num_classes: int = 10, input_size: int = 28) -> onnx.ModelProto:
    """A lightweight classifier suitable for browser inference."""
    # Flatten -> FC1 -> Relu -> FC2 (simple MLP for MNIST-like input)
    flat_dim = 1 * input_size * input_size
    hidden = 64

    W1 = numpy_helper.from_array(
        np.random.randn(flat_dim, hidden).astype(np.float32) * 0.01, name="W1"
    )
    B1 = numpy_helper.from_array(np.zeros(hidden, dtype=np.float32), name="B1")
    W2 = numpy_helper.from_array(
        np.random.randn(hidden, num_classes).astype(np.float32) * 0.01, name="W2"
    )
    B2 = numpy_helper.from_array(np.zeros(num_classes, dtype=np.float32), name="B2")

    X = helper.make_tensor_value_info("input", TensorProto.FLOAT, ["batch", 1, input_size, input_size])
    Y = helper.make_tensor_value_info("logits", TensorProto.FLOAT, ["batch", num_classes])

    flatten = helper.make_node("Flatten", ["input"], ["flat"], axis=1)
    mm1 = helper.make_node("MatMul", ["flat", "W1"], ["h1"])
    add1 = helper.make_node("Add", ["h1", "B1"], ["h1b"])
    relu = helper.make_node("Relu", ["h1b"], ["h1r"])
    mm2 = helper.make_node("MatMul", ["h1r", "W2"], ["h2"])
    add2 = helper.make_node("Add", ["h2", "B2"], ["logits"])

    graph = helper.make_graph(
        [flatten, mm1, add1, relu, mm2, add2],
        "web_classifier", [X], [Y], [W1, B1, W2, B2]
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    model.ir_version = 8
    onnx.checker.check_model(model)
    return model


web_model = build_web_classifier()
MODEL_PATH = "/tmp/web_classifier.onnx"
onnx.save(web_model, MODEL_PATH)
print(f"Web classifier: {os.path.getsize(MODEL_PATH):,} bytes")

session = ort.InferenceSession(MODEL_PATH, providers=["CPUExecutionProvider"])
test_out = session.run(None, {"input": np.random.randn(1, 1, 28, 28).astype(np.float32)})[0]
assert test_out.shape == (1, 10)
print(f"Inference OK — output shape: {test_out.shape}")

<a id='ex1'></a>
## Exercise 1 — Browser-Style Preprocessing

In browsers, image data comes from `<canvas>` as `ImageData` (RGBA, uint8, HWC).
We simulate this pipeline:

1. Extract RGBA → discard alpha → RGB (uint8)
2. Resize to model input size
3. Convert to grayscale (for MNIST-like models)
4. Normalize to $[0, 1]$ or $[-1, 1]$
5. Reshape to NCHW Float32

In [ ]:
def simulate_canvas_imagedata(width: int, height: int) -> np.ndarray:
    """Simulate browser ImageData: RGBA uint8 flat array."""
    return np.random.randint(0, 256, size=(height, width, 4), dtype=np.uint8)


def browser_preprocess(
    imagedata_rgba: np.ndarray,
    target_size: Tuple[int, int] = (28, 28),
    grayscale: bool = True,
    normalize_range: Tuple[float, float] = (0.0, 1.0),
) -> np.ndarray:
    """
    Simulate ONNX.js preprocessing pipeline.
    Input: RGBA HWC uint8 array (simulating canvas ImageData)
    Output: NCHW float32 tensor ready for inference
    """
    # Step 1: RGBA -> RGB
    rgb = imagedata_rgba[:, :, :3].astype(np.float32)

    # Step 2: Resize (nearest-neighbor, matching browser canvas behavior)
    h, w = rgb.shape[:2]
    th, tw = target_size
    row_idx = np.clip((np.arange(th) * h / th).astype(int), 0, h - 1)
    col_idx = np.clip((np.arange(tw) * w / tw).astype(int), 0, w - 1)
    resized = rgb[row_idx][:, col_idx]

    # Step 3: Grayscale conversion (luminance formula)
    if grayscale:
        gray = 0.299 * resized[:, :, 0] + 0.587 * resized[:, :, 1] + 0.114 * resized[:, :, 2]
        channels = gray[np.newaxis, :, :]  # 1, H, W
    else:
        channels = np.transpose(resized, (2, 0, 1))  # C, H, W

    # Step 4: Normalize
    lo, hi = normalize_range
    channels = channels / 255.0  # [0, 1]
    channels = channels * (hi - lo) + lo  # scale to target range

    # Step 5: Add batch dimension -> NCHW
    tensor = channels[np.newaxis, ...].astype(np.float32)
    return tensor


# Simulate browser image capture
canvas_data = simulate_canvas_imagedata(56, 56)
preprocessed = browser_preprocess(canvas_data, target_size=(28, 28))

assert preprocessed.shape == (1, 1, 28, 28)
assert preprocessed.dtype == np.float32
assert preprocessed.min() >= 0.0
assert preprocessed.max() <= 1.0
print(f"Preprocessed shape: {preprocessed.shape}, range: [{preprocessed.min():.3f}, {preprocessed.max():.3f}]")

# Run through model
output = session.run(None, {"input": preprocessed})[0]
print(f"Model output shape: {output.shape}")

<a id='ex2'></a>
## Exercise 2 — Softmax & Top-K Classification

Post-processing in the browser converts raw logits to human-readable predictions.

Numerically stable softmax:

$$p_i = \frac{e^{z_i - \max(\mathbf{z})}}{\sum_{j=1}^{K} e^{z_j - \max(\mathbf{z})}}$$

For top-k, we need $\arg\text{sort}$ descending and take first $k$ elements.

In [ ]:
def softmax(logits: np.ndarray) -> np.ndarray:
    """Numerically stable softmax (matches browser implementation)."""
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals, axis=-1, keepdims=True)


def top_k_predictions(
    logits: np.ndarray,
    k: int = 5,
    labels: Optional[List[str]] = None,
) -> List[Dict[str, Any]]:
    """Convert logits to top-k predictions with labels."""
    probs = softmax(logits.reshape(-1))
    top_indices = np.argsort(-probs)[:k]

    predictions = []
    for idx in top_indices:
        entry = {
            "index": int(idx),
            "probability": float(probs[idx]),
            "percentage": f"{probs[idx] * 100:.1f}%",
        }
        if labels and idx < len(labels):
            entry["label"] = labels[idx]
        predictions.append(entry)

    return predictions


# Test with MNIST-style labels
digit_labels = [str(i) for i in range(10)]
logits = session.run(None, {"input": preprocessed})[0]

predictions = top_k_predictions(logits, k=5, labels=digit_labels)
print("Top-5 predictions:")
for p in predictions:
    print(f"  Digit '{p['label']}': {p['percentage']}")

# Verify probabilities sum to ~1
all_probs = softmax(logits.reshape(-1))
assert abs(np.sum(all_probs) - 1.0) < 1e-5
assert len(predictions) == 5
assert predictions[0]["probability"] >= predictions[1]["probability"]
print("\nSoftmax and top-k validated.")

<a id='ex3'></a>
## Exercise 3 — Backend Selection Logic

ONNX Runtime Web supports multiple backends:

| Backend | Execution | Best For |
|---------|-----------|----------|
| **WebGPU** | GPU shaders | Large models, batch inference |
| **WebGL** | Fragment shaders | Medium models, wide compatibility |
| **WebAssembly** | CPU (SIMD) | Small models, no GPU |
| **CPU (JS)** | Pure JavaScript | Fallback only |

Selection depends on model size, device capabilities, and latency requirements.

In [ ]:
def select_web_backend(
    model_size_mb: float,
    device_capabilities: Dict[str, bool],
    latency_target_ms: float = 100.0,
) -> Dict[str, Any]:
    """
    Select optimal ONNX Runtime Web backend based on constraints.
    
    device_capabilities keys: webgpu, webgl, wasm_simd, wasm_threads
    """
    # Estimated throughput (ops/sec) per backend — simplified model
    backend_perf = {
        "webgpu": {"gflops": 2000, "overhead_ms": 5, "min_model_mb": 1.0},
        "webgl": {"gflops": 500, "overhead_ms": 10, "min_model_mb": 0.1},
        "wasm": {"gflops": 50, "overhead_ms": 2, "min_model_mb": 0.0},
    }

    # Estimate FLOPs from model size (rough: 2 FLOPs per parameter)
    est_params = model_size_mb * 1024 * 1024 / 4  # FP32 params
    est_gflops = est_params * 2 / 1e9

    candidates = []

    if device_capabilities.get("webgpu") and model_size_mb >= backend_perf["webgpu"]["min_model_mb"]:
        est_time = est_gflops / backend_perf["webgpu"]["gflops"] * 1000 + backend_perf["webgpu"]["overhead_ms"]
        candidates.append(("webgpu", est_time))

    if device_capabilities.get("webgl"):
        est_time = est_gflops / backend_perf["webgl"]["gflops"] * 1000 + backend_perf["webgl"]["overhead_ms"]
        candidates.append(("webgl", est_time))

    if device_capabilities.get("wasm_simd") or device_capabilities.get("wasm_threads"):
        multiplier = 1.0
        if device_capabilities.get("wasm_simd"):
            multiplier *= 4
        if device_capabilities.get("wasm_threads"):
            multiplier *= 2
        est_time = est_gflops / (backend_perf["wasm"]["gflops"] * multiplier) * 1000 + backend_perf["wasm"]["overhead_ms"]
        candidates.append(("wasm", est_time))

    if not candidates:
        candidates.append(("wasm", 999.0))  # Fallback

    # Sort by estimated latency
    candidates.sort(key=lambda x: x[1])
    best_backend, best_latency = candidates[0]

    return {
        "selected_backend": best_backend,
        "estimated_latency_ms": round(best_latency, 2),
        "meets_target": best_latency <= latency_target_ms,
        "all_candidates": [{"backend": b, "est_ms": round(t, 2)} for b, t in candidates],
        "model_size_mb": model_size_mb,
    }


# Test scenarios
model_size = os.path.getsize(MODEL_PATH) / (1024 * 1024)

# Modern device with WebGPU
modern = select_web_backend(
    model_size, {"webgpu": True, "webgl": True, "wasm_simd": True, "wasm_threads": True}
)
print("Modern device:", json.dumps(modern, indent=2))

# Older device (no WebGPU)
older = select_web_backend(
    model_size, {"webgpu": False, "webgl": True, "wasm_simd": True, "wasm_threads": False}
)
print("\nOlder device:", json.dumps(older, indent=2))

# Minimal device
minimal = select_web_backend(
    model_size, {"webgpu": False, "webgl": False, "wasm_simd": False, "wasm_threads": False}
)
print("\nMinimal device:", json.dumps(minimal, indent=2))

assert modern["selected_backend"] in ["webgpu", "webgl", "wasm"]
assert minimal["selected_backend"] == "wasm"

<a id='ex4'></a>
## Exercise 4 — Model Caching Strategy

Browser model loading is expensive (download + compilation). We implement
a caching strategy using content-addressable hashing:

$$\text{cache\_key} = \text{SHA256}(\text{model\_bytes})[:16]$$

This enables:
- Cache invalidation on model update
- Multiple model versions coexisting
- Offline capability via Service Worker

In [ ]:
class WebModelCache:
    """Simulates browser IndexedDB/Cache API model caching."""

    def __init__(self, max_size_mb: float = 50.0):
        self.max_size_bytes = int(max_size_mb * 1024 * 1024)
        self._cache: Dict[str, Dict[str, Any]] = {}  # key -> {bytes, metadata}
        self._access_order: List[str] = []
        self.stats = {"hits": 0, "misses": 0, "evictions": 0}

    def _hash_model(self, model_bytes: bytes) -> str:
        return hashlib.sha256(model_bytes).hexdigest()[:16]

    @property
    def used_bytes(self) -> int:
        return sum(entry["size"] for entry in self._cache.values())

    def get(self, model_url: str, model_bytes: bytes) -> Optional[str]:
        """Check if model is cached. Returns cache key if hit."""
        key = self._hash_model(model_bytes)
        if key in self._cache:
            self.stats["hits"] += 1
            self._access_order.remove(key)
            self._access_order.append(key)
            return key
        self.stats["misses"] += 1
        return None

    def put(self, model_bytes: bytes, metadata: Dict[str, Any]) -> str:
        """Store model in cache with LRU eviction."""
        key = self._hash_model(model_bytes)
        size = len(model_bytes)

        # Evict until enough space
        while self.used_bytes + size > self.max_size_bytes and self._access_order:
            evict_key = self._access_order.pop(0)
            del self._cache[evict_key]
            self.stats["evictions"] += 1

        self._cache[key] = {
            "size": size,
            "metadata": metadata,
            "cached_at": time.time(),
        }
        self._access_order.append(key)
        return key

    def status(self) -> Dict[str, Any]:
        return {
            "cached_models": len(self._cache),
            "used_mb": round(self.used_bytes / (1024 * 1024), 3),
            "max_mb": round(self.max_size_bytes / (1024 * 1024), 3),
            "utilization_pct": round(100 * self.used_bytes / self.max_size_bytes, 1),
            "stats": self.stats,
        }


# Test caching
cache = WebModelCache(max_size_mb=0.5)

with open(MODEL_PATH, "rb") as f:
    model_bytes = f.read()

# First access: miss
result = cache.get("https://cdn.example.com/model.onnx", model_bytes)
assert result is None

# Store in cache
key = cache.put(model_bytes, {"version": "1.0", "url": "https://cdn.example.com/model.onnx"})
print(f"Cached with key: {key}")

# Second access: hit
result = cache.get("https://cdn.example.com/model.onnx", model_bytes)
assert result == key

# Different model: miss
other_bytes = model_bytes + b"\x00"
result = cache.get("https://cdn.example.com/model_v2.onnx", other_bytes)
assert result is None

print("\nCache status:", json.dumps(cache.status(), indent=2))
assert cache.stats["hits"] == 1
assert cache.stats["misses"] == 2

<a id='ex5'></a>
## Exercise 5 — Model Size Optimization for Web

Web delivery has strict size budgets. Users won't wait for large downloads.

Target sizes for web:
- **Ideal:** < 1 MB (instant load on 4G)
- **Acceptable:** 1–5 MB (few seconds on 4G)
- **Large:** > 5 MB (needs loading indicator)

Download time estimation:

$$t_{\text{download}} = \frac{\text{model\_size\_bytes}}{\text{bandwidth\_Bps}} + \text{RTT}$$

In [ ]:
def analyze_web_delivery(
    model_path: str,
    bandwidth_profiles: Optional[Dict[str, float]] = None,
) -> Dict[str, Any]:
    """Analyze model delivery characteristics for web."""
    if bandwidth_profiles is None:
        bandwidth_profiles = {
            "4g": 5_000_000,    # 5 MB/s
            "3g": 500_000,      # 500 KB/s
            "wifi": 25_000_000, # 25 MB/s
            "edge": 50_000,     # 50 KB/s
        }

    file_size = os.path.getsize(model_path)

    # Estimate gzip compression (ONNX models typically compress 30-60%)
    model = onnx.load(model_path)
    n_params = sum(
        int(np.prod(list(init.dims))) for init in model.graph.initializer
    )
    est_gzip_ratio = 0.6  # conservative
    gzip_size = int(file_size * est_gzip_ratio)

    download_times = {}
    for name, bw in bandwidth_profiles.items():
        rtt_ms = {"4g": 50, "3g": 200, "wifi": 10, "edge": 500}.get(name, 100)
        time_ms = (gzip_size / bw) * 1000 + rtt_ms
        download_times[name] = round(time_ms, 0)

    # Size rating
    size_mb = file_size / (1024 * 1024)
    if size_mb < 1:
        rating = "excellent"
    elif size_mb < 5:
        rating = "acceptable"
    else:
        rating = "large"

    return {
        "raw_size_bytes": file_size,
        "raw_size_kb": round(file_size / 1024, 2),
        "est_gzip_bytes": gzip_size,
        "est_gzip_kb": round(gzip_size / 1024, 2),
        "parameters": n_params,
        "download_time_ms": download_times,
        "size_rating": rating,
        "needs_loading_indicator": any(t > 2000 for t in download_times.values()),
    }


delivery = analyze_web_delivery(MODEL_PATH)
print("Web Delivery Analysis:")
print(json.dumps(delivery, indent=2))

assert delivery["size_rating"] in ["excellent", "acceptable", "large"]
assert delivery["est_gzip_bytes"] < delivery["raw_size_bytes"]
print(f"\nSize rating: {delivery['size_rating']}")

<a id='ex6'></a>
## Exercise 6 — JSON-Serializable Inference API

Browser inference APIs communicate via JSON. We build a complete
request/response schema that could be used with Web Workers.

In [ ]:
class WebInferenceAPI:
    """JSON-based inference API matching ONNX Runtime Web patterns."""

    def __init__(self, session: ort.InferenceSession):
        self.session = session

    def get_model_info(self) -> Dict[str, Any]:
        """Return model metadata as JSON-serializable dict."""
        inputs = []
        for inp in self.session.get_inputs():
            inputs.append({
                "name": inp.name,
                "shape": [str(d) for d in inp.shape],
                "type": inp.type,
            })
        outputs = []
        for out in self.session.get_outputs():
            outputs.append({
                "name": out.name,
                "shape": [str(d) for d in out.shape],
                "type": out.type,
            })
        return {"inputs": inputs, "outputs": outputs}

    def infer(self, request_json: str) -> str:
        """Accept JSON string, return JSON string (simulating postMessage)."""
        request = json.loads(request_json)

        # Build feeds from request
        feeds = {}
        for tensor_spec in request.get("inputs", []):
            name = tensor_spec["name"]
            data = np.array(tensor_spec["data"], dtype=np.float32)
            shape = tensor_spec["shape"]
            feeds[name] = data.reshape(shape)

        # Run inference
        output_names = [o.name for o in self.session.get_outputs()]
        outputs = self.session.run(output_names, feeds)

        # Serialize response
        response = {"outputs": []}
        for name, arr in zip(output_names, outputs):
            response["outputs"].append({
                "name": name,
                "shape": list(arr.shape),
                "data": arr.ravel().tolist(),
            })

        return json.dumps(response)


api = WebInferenceAPI(session)

# Get model info
info = api.get_model_info()
print("Model Info:", json.dumps(info, indent=2))

# Simulate Web Worker message passing
request = json.dumps({
    "inputs": [{
        "name": "input",
        "shape": [1, 1, 28, 28],
        "data": np.random.randn(1 * 1 * 28 * 28).tolist(),
    }]
})

response_json = api.infer(request)
response = json.loads(response_json)

assert len(response["outputs"]) == 1
assert response["outputs"][0]["shape"] == [1, 10]
assert len(response["outputs"][0]["data"]) == 10
print(f"\nInference response output shape: {response['outputs'][0]['shape']}")
print("JSON round-trip validated.")

<a id='ex7'></a>
## Exercise 7 — Float16 & Quantized Models for Web

WebGPU natively supports Float16, making FP16 models ideal for GPU-accelerated
web inference. Quantized (INT8) models are best for WebAssembly.

Size comparison:

| Precision | Bytes/Param | Relative Size |
|:---------:|:-----------:|:-------------:|
| FP32 | 4 | 1.0× |
| FP16 | 2 | 0.5× |
| INT8 | 1 | 0.25× |

In [ ]:
def convert_to_float16(model_path: str, output_path: str) -> Dict[str, Any]:
    """Convert FP32 initializers to FP16 (simulating onnxconverter-common)."""
    model = onnx.load(model_path)

    fp32_size = 0
    fp16_size = 0

    for init in model.graph.initializer:
        if init.data_type == TensorProto.FLOAT:
            arr = numpy_helper.to_array(init)
            fp32_size += arr.nbytes
            arr_fp16 = arr.astype(np.float16)
            fp16_size += arr_fp16.nbytes

            new_init = numpy_helper.from_array(arr_fp16, name=init.name)
            init.CopyFrom(new_init)

    # Update input types to match (for this demo, keep inputs as FP32)
    onnx.save(model, output_path)

    return {
        "original_param_bytes": fp32_size,
        "fp16_param_bytes": fp16_size,
        "compression": round(fp32_size / max(fp16_size, 1), 2),
        "original_file_bytes": os.path.getsize(model_path),
        "fp16_file_bytes": os.path.getsize(output_path),
    }


from onnxruntime.quantization import quantize_dynamic, QuantType

FP16_PATH = "/tmp/web_classifier_fp16.onnx"
INT8_PATH = "/tmp/web_classifier_int8.onnx"

# FP16 conversion
fp16_info = convert_to_float16(MODEL_PATH, FP16_PATH)
print("FP16 Conversion:")
print(json.dumps(fp16_info, indent=2))

# INT8 quantization
quantize_dynamic(model_input=MODEL_PATH, model_output=INT8_PATH, weight_type=QuantType.QUInt8)

# Compare all variants
fp32_size = os.path.getsize(MODEL_PATH)
fp16_size = os.path.getsize(FP16_PATH)
int8_size = os.path.getsize(INT8_PATH)

print(f"\nSize Comparison:")
print(f"  FP32: {fp32_size:>8,} bytes (1.00×)")
print(f"  FP16: {fp16_size:>8,} bytes ({fp32_size/fp16_size:.2f}×)")
print(f"  INT8: {int8_size:>8,} bytes ({fp32_size/int8_size:.2f}×)")

assert fp16_size < fp32_size
print("\nAll web variants generated successfully.")

<a id='challenge'></a>
## Challenge — Web Deployment Manifest Generator

Create a complete web deployment manifest that includes:
- Model variants (FP32, FP16, INT8) with selection rules
- Preprocessing specification
- Backend recommendations per device class
- CDN delivery configuration

In [ ]:
def generate_web_deployment_manifest(
    model_path: str,
    app_name: str = "digit-classifier",
    version: str = "1.0.0",
    cdn_base: str = "https://cdn.example.com/models",
) -> Dict[str, Any]:
    """Generate complete web deployment manifest."""
    # Analyze all variants
    variants = {}
    for variant_name, path in [("fp32", MODEL_PATH), ("fp16", FP16_PATH), ("int8", INT8_PATH)]:
        with open(path, "rb") as f:
            content = f.read()
        variants[variant_name] = {
            "filename": f"{app_name}_{variant_name}.onnx",
            "size_bytes": len(content),
            "sha256": hashlib.sha256(content).hexdigest(),
            "url": f"{cdn_base}/{app_name}/{version}/{app_name}_{variant_name}.onnx",
        }

    manifest = {
        "manifest_version": "2.0",
        "app": app_name,
        "model_version": version,
        "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "variants": variants,
        "backend_rules": [
            {
                "condition": "webgpu_available",
                "variant": "fp16",
                "backend": "webgpu",
                "reason": "WebGPU has native FP16 support",
            },
            {
                "condition": "webgl_available && !webgpu_available",
                "variant": "fp32",
                "backend": "webgl",
                "reason": "WebGL prefers FP32 textures",
            },
            {
                "condition": "wasm_simd_available",
                "variant": "int8",
                "backend": "wasm",
                "reason": "WASM SIMD accelerates INT8 ops",
            },
            {
                "condition": "default",
                "variant": "int8",
                "backend": "wasm",
                "reason": "Smallest download, broadest compatibility",
            },
        ],
        "preprocessing": {
            "input_source": "canvas_imagedata",
            "steps": [
                {"op": "rgba_to_gray", "method": "luminance"},
                {"op": "resize", "target": [28, 28], "method": "nearest"},
                {"op": "normalize", "range": [0.0, 1.0]},
                {"op": "reshape", "shape": [1, 1, 28, 28]},
            ],
        },
        "postprocessing": {
            "steps": [
                {"op": "softmax"},
                {"op": "top_k", "k": 5},
                {"op": "map_labels", "labels": [str(i) for i in range(10)]},
            ],
        },
        "caching": {
            "strategy": "content-addressable",
            "storage": "indexeddb",
            "max_cache_mb": 50,
            "ttl_days": 30,
        },
    }
    return manifest


manifest = generate_web_deployment_manifest(MODEL_PATH)
print(json.dumps(manifest, indent=2))

# Validate manifest structure
assert "variants" in manifest
assert len(manifest["variants"]) == 3
assert all(v["sha256"] for v in manifest["variants"].values())
assert len(manifest["backend_rules"]) >= 3
assert manifest["preprocessing"]["input_source"] == "canvas_imagedata"
print("\nWeb deployment manifest validated!")

<a id='summary'></a>
## Summary

| Component | Web-Specific Consideration |
|-----------|---------------------------|
| **Preprocessing** | Canvas ImageData (RGBA) → model tensor |
| **Postprocessing** | Softmax + top-k for user display |
| **Backend Selection** | WebGPU > WebGL > WASM (fallback chain) |
| **Caching** | Content-addressable, IndexedDB storage |
| **Size Budget** | < 1MB ideal, gzip helps 30-60% |
| **JSON API** | Web Workers need serializable messages |
| **Precision** | FP16 for WebGPU, INT8 for WASM |

**Key insight:** Web deployment optimizes for *initial load time* and *device compatibility*,
not just inference speed. The backend selection cascade ensures every user gets the best
experience their hardware supports.